<div style="background-color:#ED1C24; color:white; padding:20px 30px; border-radius:10px;">
<h1 style="color:white; margin:0;">Session 9.2: Logistic Regression & Classification</h1>
<p style="color:white; margin:5px 0 0 0;">Practical Lab — From "How Much?" to "Which Category?"</p>
</div>

**Program:** Vishlesan i-Hub IIT Patna x Masai School -- AIM (AI & Machine Learning)
**Session ID:** 9.2 | **Week:** 9
**Prerequisites:** Session 9.1 (KNN & Simple Linear Regression)

## Learning Objectives

By the end of this notebook, you will be able to:

1. Explain WHY linear regression fails for classification (outputs outside [0,1])
2. Describe the **sigmoid function** and how it converts linear output to probabilities
3. Build **logistic regression** with sklearn (`fit`, `predict`, `predict_proba`)
4. Construct and interpret a **confusion matrix** (TP, FP, FN, TN)
5. Compute and interpret **precision**, **recall**, and **F1-score**
6. Explain the **precision-recall tradeoff** and when to prioritize each
7. Tune the **decision threshold** to optimize for business objectives

## Prerequisites

You should be comfortable with:
- **sklearn API**: `fit()`, `predict()`, `score()`, train/test split, Pipeline (Session 8.2)
- **Linear Regression**: y = mx + b, RMSE, R-squared (Session 9.1)
- **Model comparison**: evaluating two models on the same data (Session 9.1)
- **Type I and Type II errors** from hypothesis testing (Session 7.2)
- **Probability** basics (Sessions 6.2-7.1)

---

## Setup

In [51]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                              precision_score, recall_score, f1_score)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.datasets import load_breast_cancer, make_classification
from scipy.special import expit

In [52]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "colab"
np.random.seed(42)
print("Setup complete! Classification tools ready.")

Setup complete! Classification tools ready.


In [53]:
from IPython.display import HTML, display

_BOX_STYLES = {
    "definition": ("#448aff", "#e3f2fd", "#1565c0"),
    "tip":        ("#00c853", "#e8f5e9", "#2e7d32"),
    "warning":    ("#ff9100", "#fff3e0", "#e65100"),
    "danger":     ("#ff1744", "#fce4ec", "#c62828"),
    "math":       ("#7c4dff", "#ede7f6", "#4527a0"),
    "output":     ("#00b8d4", "#e0f7fa", "#006064"),
    "industry":   ("#009688", "#e0f2f1", "#004d40"),
}

In [54]:
def box(kind, title, content):
    """Display a styled callout box."""
    border, bg, title_clr = _BOX_STYLES[kind]
    display(HTML(f"""
    <div style="margin:12px 0; padding:12px 16px; border-left:4px solid {border};
                background-color:{bg}; border-radius:4px;">
    <strong style="color:{title_clr};">{title}</strong><br>{content}
    </div>"""))

---

## Bridge from Session 9.1

In Session 9.1, you asked: **"What is the VALUE?"** — predicting continuous numbers (house prices, test scores). Every prediction was a number on a continuous scale.

Today's question is different: **"Which CATEGORY?"** — Is this email spam or not? Is this tumor malignant or benign? Will this student pass or fail?

These are **classification** problems. The target is not a number — it is a **category**.

---

## Business Context: Email Spam Detection

Gmail processes **billions of emails daily** and must classify each as spam or not-spam in milliseconds.

- **False Positive** (FP): A real email marked as spam — you miss an important message from your boss
- **False Negative** (FN): Spam gets through — annoying but usually harmless

Google optimizes for **high precision** (never send a real email to spam) while maintaining acceptable recall. The balance between these metrics is the central challenge of classification.

Today you learn the tools to build and evaluate classifiers like Gmail's.

In [55]:
box("industry", "Gmail's Spam Classification",
    "Gmail's spam filter uses logistic regression (among other models) to assign a "
    "spam probability to every email. The system processes billions of emails daily "
    "and must balance precision (don't block real email) with recall (catch spam). "
    "Today you learn the exact metrics Gmail's engineers use to evaluate their models.")

---

<div style="background-color:#ED1C24; color:white; padding:15px 25px; border-radius:8px; margin:20px 0;">
<h2 style="color:white; margin:0;">Part 1: Why Not Linear Regression for Classification?</h2>
</div>

### The Rain Probability Analogy

Can the probability of rain be **-20%**? Can it be **130%**? No — probability must be between 0 and 1. Linear regression does not know this constraint. It happily predicts impossible values.

We need a model that **respects the rules of probability**.

Let's see what happens when we use LinearRegression on binary labels (0 = fail, 1 = pass):

In [56]:
# Generate study hours → pass/fail data
np.random.seed(42)
n = 40
study_hours = np.sort(np.random.uniform(1, 10, n))
prob_pass = 1 / (1 + np.exp(-(study_hours - 5)))
y_pass = (np.random.random(n) < prob_pass).astype(int)
X_study = study_hours.reshape(-1, 1)

print(f"Dataset: {n} students")
print(f"Study hours range: {study_hours.min():.1f} to {study_hours.max():.1f}")
print(f"Pass rate: {y_pass.mean():.1%}")

Dataset: 40 students
Study hours range: 1.2 to 9.7
Pass rate: 55.0%


In [57]:
# Fit LINEAR regression on binary data (WRONG approach)
lr_wrong = LinearRegression()
lr_wrong.fit(X_study, y_pass)

print("=== LINEAR REGRESSION ON BINARY LABELS ===")
print(f"At 0 hours:  {lr_wrong.predict([[0]])[0]:.3f}  <- NEGATIVE probability?!")
print(f"At 5 hours:  {lr_wrong.predict([[5]])[0]:.3f}")
print(f"At 12 hours: {lr_wrong.predict([[12]])[0]:.3f}  <- Over 100%?!")

=== LINEAR REGRESSION ON BINARY LABELS ===
At 0 hours:  -0.189  <- NEGATIVE probability?!
At 5 hours:  0.535
At 12 hours: 1.547  <- Over 100%?!


In [58]:
box("danger", "Linear Regression Fails for Classification",
    "Linear regression outputs any real number from -inf to +inf. "
    "For classification, we need probabilities in [0, 1]. "
    "A prediction of -0.15 or 1.3 is meaningless as a probability.<br><br>"
    "<b>Solution:</b> Logistic regression wraps the linear output in a sigmoid function "
    "that squashes it into [0, 1].")

Now let's fit **LogisticRegression** on the same data and compare:

In [59]:
# Fit LOGISTIC regression (CORRECT approach)
log_reg = LogisticRegression(random_state=42)
log_reg.fit(X_study, y_pass)

print("=== LOGISTIC REGRESSION (CORRECT) ===")
print(f"At 0 hours:  {log_reg.predict_proba([[0]])[0,1]:.3f}  <- Valid probability!")
print(f"At 5 hours:  {log_reg.predict_proba([[5]])[0,1]:.3f}")
print(f"At 12 hours: {log_reg.predict_proba([[12]])[0,1]:.3f}  <- Valid probability!")

=== LOGISTIC REGRESSION (CORRECT) ===
At 0 hours:  0.007  <- Valid probability!
At 5 hours:  0.643
At 12 hours: 1.000  <- Valid probability!


The interactive chart below shows both models overlaid on the same data. The **red dashed line** (linear regression) goes outside [0,1] — broken. The **green S-curve** (logistic regression) stays within [0,1] — correct. Hover over any point to see its actual label.

In [60]:
# INTERACTIVE: Linear vs Logistic comparison
X_range = np.linspace(0, 12, 200).reshape(-1, 1)
y_lr = lr_wrong.predict(X_range)
y_log = log_reg.predict_proba(X_range)[:, 1]
colors = ["#E74C3C" if y==0 else "#2ECC71" for y in y_pass]

fig = go.Figure()
fig.add_trace(go.Scatter(x=study_hours, y=y_pass, mode="markers",
    marker=dict(color=colors, size=10, line=dict(color="black", width=0.5)),
    name="Actual (0=Fail, 1=Pass)",
    hovertemplate="Hours: %{x:.1f}<br>Result: %{y}<extra></extra>"))

In [61]:
fig.add_trace(go.Scatter(x=X_range.flatten(), y=y_lr, mode="lines",
    line=dict(color="#E74C3C", width=2, dash="dash"), name="Linear Reg (BROKEN)"))
fig.add_trace(go.Scatter(x=X_range.flatten(), y=y_log, mode="lines",
    line=dict(color="#2ECC71", width=3), name="Logistic Reg (CORRECT)"))
fig.add_hrect(y0=-0.3, y1=0, fillcolor="red", opacity=0.05, line_width=0)
fig.add_hrect(y0=1, y1=1.3, fillcolor="red", opacity=0.05, line_width=0)
fig.update_layout(title="Why Not Linear Regression for Classification?",
    xaxis_title="Study Hours", yaxis_title="P(Pass)",
    yaxis=dict(range=[-0.3, 1.3]), width=850, height=500)
fig.show()

In [62]:
box("definition", "Classification vs Regression",
    "<b>Regression:</b> predict a continuous value (price, temperature, score). "
    "Use LinearRegression, evaluate with RMSE/R-squared.<br>"
    "<b>Classification:</b> predict a category (spam/not-spam, pass/fail). "
    "Use LogisticRegression, evaluate with confusion matrix, precision, recall.<br><br>"
    "Despite its name, logistic regression is a <b>classification</b> algorithm.")

---

<div style="background-color:#ED1C24; color:white; padding:15px 25px; border-radius:8px; margin:20px 0;">
<h2 style="color:white; margin:0;">Part 2: The Sigmoid Function</h2>
</div>

### The Surgeon's Confidence Dial

Imagine a surgeon deciding whether to operate. They look at all the patient's indicators. Lots of bad signs — the dial reads near **0% (don't operate)**. Lots of good signs — near **100% (operate)**. Mixed signals — around **50% (uncertain)**.

The dial physically CANNOT go below 0% or above 100%. That is exactly what the sigmoid function does.

In [63]:
box("definition", "The Sigmoid Function",
    "sigma(z) = 1 / (1 + e<sup>-z</sup>)<br><br>"
    "Takes ANY real number z from -inf to +inf and outputs a value in (0, 1).<br>"
    "This converts a linear score into a valid probability.<br><br>"
    "<b>Key points:</b><br>"
    "sigma(-5) approx 0.007 — very confident class 0<br>"
    "sigma(0) = 0.5 — completely uncertain (coin flip)<br>"
    "sigma(+5) approx 0.993 — very confident class 1")

### Step-by-Step: Computing the Sigmoid Manually

In [64]:
# Manual sigmoid computation
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z_values = np.array([-5, -2, 0, 2, 5])
sig_values = sigmoid(z_values)

print("=== SIGMOID: Step by Step ===\n")
print(f"  z   |  e^(-z)  | 1+e^(-z) | 1/(1+e^(-z))")
print("-" * 48)
for z in z_values:
    e_neg_z = np.exp(-z)
    denom = 1 + e_neg_z
    result = 1 / denom
    print(f"{z:>5.1f} | {e_neg_z:>8.4f} | {denom:>8.4f} | {result:>12.4f}")

=== SIGMOID: Step by Step ===

  z   |  e^(-z)  | 1+e^(-z) | 1/(1+e^(-z))
------------------------------------------------
 -5.0 | 148.4132 | 149.4132 |       0.0067
 -2.0 |   7.3891 |   8.3891 |       0.1192
  0.0 |   1.0000 |   2.0000 |       0.5000
  2.0 |   0.1353 |   1.1353 |       0.8808
  5.0 |   0.0067 |   1.0067 |       0.9933


In [65]:
# Verify with scipy
print("\nVerify with scipy.special.expit:")
print(f"  sigmoid(2.0) = {sigmoid(2.0):.6f}")
print(f"  expit(2.0)   = {expit(2.0):.6f}")
print(f"  Match: {np.isclose(sigmoid(2.0), expit(2.0))}")


Verify with scipy.special.expit:
  sigmoid(2.0) = 0.880797
  expit(2.0)   = 0.880797
  Match: True


The interactive chart below is the **Sigmoid Explorer** — the S-curve that converts any number to a probability. Hover over any point to see its exact value.

In [66]:
# INTERACTIVE: Sigmoid Explorer (SIGNATURE VISUALIZATION)
z = np.linspace(-8, 8, 300)
s = sigmoid(z)

fig = go.Figure()
fig.add_trace(go.Scatter(x=z, y=s, mode="lines",
    line=dict(color="#3498DB", width=3), name="sigmoid(z)",
    hovertemplate="z = %{x:.2f}<br>sigma(z) = %{y:.4f}<extra></extra>"))
for zp in [-5, 0, 5]:
    fig.add_trace(go.Scatter(x=[zp], y=[sigmoid(zp)], mode="markers",
        marker=dict(color="#ED1C24", size=12), showlegend=False))

In [67]:
fig.add_hline(y=0.5, line_dash="dash", line_color="gray", opacity=0.5)
fig.add_hline(y=0, line_dash="dot", line_color="gray", opacity=0.3)
fig.add_hline(y=1, line_dash="dot", line_color="gray", opacity=0.3)
fig.update_layout(title="Sigmoid Explorer: Any Number -> Probability",
    xaxis_title="z (linear combination: wx + b)", yaxis_title="sigma(z) = P(y=1)",
    yaxis=dict(range=[-0.05, 1.05]), width=800, height=500)
fig.show()

In [68]:
box("math", "Sigmoid Formula",
    "sigma(z) = 1 / (1 + e<sup>-z</sup>)<br><br>"
    "In logistic regression, z = w*x + b (same linear combination as linear regression).<br>"
    "The sigmoid wraps around it: P(y=1) = sigma(w*x + b)<br><br>"
    "Two steps: (1) compute linear score z = wx + b, (2) squash through sigmoid to get probability.")

---

<div style="background-color:#ED1C24; color:white; padding:15px 25px; border-radius:8px; margin:20px 0;">
<h2 style="color:white; margin:0;">Part 3: LogisticRegression in sklearn + Log Loss</h2>
</div>

In [69]:
box("math", "sklearn API: predict() vs predict_proba()",
    "<code>model = LogisticRegression()</code><br>"
    "<code>model.fit(X_train, y_train)</code><br>"
    "<code>model.predict(X_test)</code> — returns labels: [0, 1, 1, 0, ...]<br>"
    "<code>model.predict_proba(X_test)</code> — returns probabilities: [[0.9, 0.1], [0.3, 0.7], ...]<br><br>"
    "<b>Key:</b> predict_proba() is MORE useful — it tells you HOW confident the model is, not just its answer.")

Let's build a classifier on 2D synthetic data and see both outputs:

In [70]:
# Generate 2D classification data
X_cls, y_cls = make_classification(n_samples=200, n_features=2, n_redundant=0,
    n_informative=2, random_state=42, n_clusters_per_class=1)
X_tr, X_te, y_tr, y_te = train_test_split(X_cls, y_cls, test_size=0.2, random_state=42)

model = LogisticRegression(random_state=42)
model.fit(X_tr, y_tr)

y_pred = model.predict(X_te)
y_proba = model.predict_proba(X_te)

print("=== predict() vs predict_proba() ===\n")
print("First 5 test samples:")
print(f"  predict():      {y_pred[:5]}")
print(f"  predict_proba():")
for i in range(5):
    print(f"    Sample {i}: P(0)={y_proba[i,0]:.3f}, P(1)={y_proba[i,1]:.3f} -> label={y_pred[i]}")

=== predict() vs predict_proba() ===

First 5 test samples:
  predict():      [0 1 1 1 1]
  predict_proba():
    Sample 0: P(0)=0.902, P(1)=0.098 -> label=0
    Sample 1: P(0)=0.302, P(1)=0.698 -> label=1
    Sample 2: P(0)=0.299, P(1)=0.701 -> label=1
    Sample 3: P(0)=0.362, P(1)=0.638 -> label=1
    Sample 4: P(0)=0.190, P(1)=0.810 -> label=1


### Log Loss: The Cost of Being Confident and Wrong

What happens when the model is very confident — and very WRONG?

In [71]:
# Log loss catastrophic penalty
print("=== LOG LOSS: Confident & Wrong = Catastrophic ===\n")
print(f"  True label = 1 (has disease)")
print(f"  {'Prediction':>12} {'Loss':>8} {'Verdict':>15}")
print(f"  {'-'*38}")
cases = [(0.99, "Good"), (0.90, "OK"), (0.50, "Uncertain"), (0.10, "Bad"), (0.01, "CATASTROPHIC")]
for p, verdict in cases:
    loss = -np.log(p)
    print(f"  p = {p:>8.2f}   {loss:>8.4f}   {verdict:>15}")
print(f"\n  Ratio: p=0.01 is {-np.log(0.01)/-np.log(0.99):.0f}x worse than p=0.99!")

=== LOG LOSS: Confident & Wrong = Catastrophic ===

  True label = 1 (has disease)
    Prediction     Loss         Verdict
  --------------------------------------
  p =     0.99     0.0101              Good
  p =     0.90     0.1054                OK
  p =     0.50     0.6931         Uncertain
  p =     0.10     2.3026               Bad
  p =     0.01     4.6052      CATASTROPHIC

  Ratio: p=0.01 is 458x worse than p=0.99!


The interactive chart below shows the log loss function. Notice the **cliff** as the predicted probability approaches 0 when the true label is 1 — the penalty explodes.

In [72]:
# INTERACTIVE: Log Loss Cliff
p = np.linspace(0.01, 0.99, 200)
loss = -np.log(p)

fig = go.Figure()
fig.add_trace(go.Scatter(x=p, y=loss, mode="lines", fill="tozeroy",
    line=dict(color="#E74C3C", width=2), fillcolor="rgba(231,76,60,0.1)",
    name="Loss = -log(p)", hovertemplate="p = %{x:.2f}<br>Loss = %{y:.3f}<extra></extra>"))
for pp in [0.01, 0.1, 0.5, 0.9, 0.99]:
    fig.add_trace(go.Scatter(x=[pp], y=[-np.log(pp)], mode="markers",
        marker=dict(size=10, color="#2C3E50"), showlegend=False,
        hovertemplate=f"p={pp:.2f}<br>Loss={-np.log(pp):.3f}<extra></extra>"))

In [73]:
fig.add_vrect(x0=0, x1=0.2, fillcolor="red", opacity=0.08, line_width=0,
    annotation_text="DANGER ZONE", annotation_position="top left")
fig.update_layout(title="Log Loss: Confident Wrong Predictions (true label = 1)",
    xaxis_title="Predicted Probability p", yaxis_title="Loss = -log(p)",
    width=800, height=450)
fig.show()

In [74]:
box("warning", "Confident AND Wrong = Catastrophic",
    "Log loss penalizes confident wrong predictions SEVERELY.<br>"
    "p=0.99 with y=1 -> loss = 0.01 (barely penalized)<br>"
    "p=0.01 with y=1 -> loss = 4.61 (460x worse!)<br><br>"
    "This is why logistic regression learns to be well-calibrated — "
    "it avoids extreme confidence unless the evidence is strong.")

---

<div style="background-color:#ED1C24; color:white; padding:15px 25px; border-radius:8px; margin:20px 0;">
<h2 style="color:white; margin:0;">Part 4: Confusion Matrix — The Classifier's Report Card</h2>
</div>

### Three Real-World Analogies

**COVID Testing:**
- TP: Sick person, test says sick (correct!)
- FP: Healthy person, test says sick (unnecessary quarantine)
- FN: Sick person, test says healthy (spreads virus!)
- TN: Healthy person, test says healthy (correct!)

**Spam Filter:**
- TP: Spam caught by filter (correct!)
- FP: Real email marked as spam (miss important message!)
- FN: Spam gets through (annoying)
- TN: Real email delivered (correct!)

**Which error is worse depends on context.** For COVID: FN is catastrophic. For spam: FP is worse.

In [75]:
box("definition", "Confusion Matrix: TP, FP, FN, TN",
    "A 2x2 grid that breaks down ALL predictions into 4 categories:<br><br>"
    "<b>True Positive (TP):</b> Predicted 1, actually 1 (correct!)<br>"
    "<b>False Positive (FP):</b> Predicted 1, actually 0 (false alarm — Type I Error)<br>"
    "<b>False Negative (FN):</b> Predicted 0, actually 1 (missed case — Type II Error)<br>"
    "<b>True Negative (TN):</b> Predicted 0, actually 0 (correct!)")

### Step-by-Step: Counting TP, FP, FN, TN Manually

In [76]:
# Manual confusion matrix computation
y_true = np.array([1, 0, 1, 1, 0, 1, 0, 0, 1, 0])
y_pred_ex = np.array([1, 0, 1, 0, 0, 1, 1, 0, 0, 0])

tp = np.sum((y_pred_ex == 1) & (y_true == 1))
fp = np.sum((y_pred_ex == 1) & (y_true == 0))
fn = np.sum((y_pred_ex == 0) & (y_true == 1))
tn = np.sum((y_pred_ex == 0) & (y_true == 0))

print("=== MANUAL CONFUSION MATRIX ===\n")
print(f"y_true: {y_true}")
print(f"y_pred: {y_pred_ex}\n")
print(f"True Positives (TP):  {tp}  <- Correctly predicted 1")
print(f"False Positives (FP): {fp}  <- Predicted 1, was actually 0")
print(f"False Negatives (FN): {fn}  <- Predicted 0, was actually 1")
print(f"True Negatives (TN):  {tn}  <- Correctly predicted 0")

=== MANUAL CONFUSION MATRIX ===

y_true: [1 0 1 1 0 1 0 0 1 0]
y_pred: [1 0 1 0 0 1 1 0 0 0]

True Positives (TP):  3  <- Correctly predicted 1
False Positives (FP): 1  <- Predicted 1, was actually 0
False Negatives (FN): 2  <- Predicted 0, was actually 1
True Negatives (TN):  4  <- Correctly predicted 0


In [77]:
# sklearn does this in one line
cm = confusion_matrix(y_true, y_pred_ex)
print("sklearn confusion_matrix():")
print(cm)
print(f"\nVerify: TN={cm[0,0]}, FP={cm[0,1]}, FN={cm[1,0]}, TP={cm[1,1]}")

sklearn confusion_matrix():
[[4 1]
 [2 3]]

Verify: TN=4, FP=1, FN=2, TP=3


Now let's compute the confusion matrix for our logistic regression model on the test data:

In [78]:
# Confusion matrix for our model
cm_model = confusion_matrix(y_te, y_pred)
tn_m, fp_m, fn_m, tp_m = cm_model.ravel()

print("=== MODEL CONFUSION MATRIX ===\n")
print(f"               Predicted 0  Predicted 1")
print(f"  Actual 0:      {tn_m:>4} (TN)    {fp_m:>4} (FP)")
print(f"  Actual 1:      {fn_m:>4} (FN)    {tp_m:>4} (TP)")
print(f"\n  Accuracy: {accuracy_score(y_te, y_pred):.1%}")

=== MODEL CONFUSION MATRIX ===

               Predicted 0  Predicted 1
  Actual 0:        18 (TN)       5 (FP)
  Actual 1:         0 (FN)      17 (TP)

  Accuracy: 87.5%


The interactive heatmap below shows the confusion matrix visually. Hover over each cell to see its meaning.

In [79]:
# INTERACTIVE: Confusion Matrix Heatmap
labels = [["TN (Correct Reject)", "FP (False Alarm)"],
          ["FN (Missed Case)", "TP (Correct Detection)"]]
hover = [[f"True Neg: {tn_m}<br>Predicted 0, Actually 0", f"False Pos: {fp_m}<br>Predicted 1, Actually 0"],
         [f"False Neg: {fn_m}<br>Predicted 0, Actually 1", f"True Pos: {tp_m}<br>Predicted 1, Actually 1"]]
fig = go.Figure(go.Heatmap(z=cm_model, x=["Predicted 0","Predicted 1"],
    y=["Actual 0","Actual 1"], colorscale="Blues", showscale=False,
    text=[[f"{tn_m}\n(TN)", f"{fp_m}\n(FP)"], [f"{fn_m}\n(FN)", f"{tp_m}\n(TP)"]],
    texttemplate="%{text}", textfont=dict(size=16),
    customdata=hover, hovertemplate="%{customdata}<extra></extra>"))
fig.update_layout(title="Confusion Matrix", width=500, height=450,
    xaxis_title="Model Prediction", yaxis_title="Ground Truth")
fig.show()

In [80]:
box("tip", "Connection to Session 7.2",
    "<b>Type I Error = False Positive (FP)</b> — false alarm, rejecting H0 when it is true.<br>"
    "<b>Type II Error = False Negative (FN)</b> — missed detection, failing to reject H0 when it is false.<br><br>"
    "The confusion matrix COUNTS these errors from your model's predictions.")

---

<div style="background-color:#ED1C24; color:white; padding:15px 25px; border-radius:8px; margin:20px 0;">
<h2 style="color:white; margin:0;">Part 5: Precision vs Recall — The Classification Tradeoff</h2>
</div>

### The Fishing Net Analogy

Imagine a lake with **1000 fish**: 300 goldfish (your target) + 700 bass (not wanted).

Your net catches **200 fish**: 180 goldfish + 20 bass.

- **Precision = 180/200 = 90%** — "Of the fish I caught, 90% are goldfish." How **pure** is my catch?
- **Recall = 180/300 = 60%** — "Of all goldfish in the lake, I caught 60%." How **complete** is my search?

**The tradeoff:** A finer net catches only big goldfish (precision UP, recall DOWN). A coarser net catches everything (recall UP, precision DOWN).

In [81]:
box("definition", "Precision and Recall",
    "<b>Precision = TP / (TP + FP)</b><br>"
    "Of all POSITIVE predictions, how many were correct?<br>"
    "Question: How many false alarms did I give?<br><br>"
    "<b>Recall = TP / (TP + FN)</b><br>"
    "Of all ACTUAL positives, how many did I find?<br>"
    "Question: How many real cases did I miss?")

### Step-by-Step: Computing Precision and Recall Manually

In [82]:
# Manual computation from our model's confusion matrix
precision_manual = tp_m / (tp_m + fp_m)
recall_manual = tp_m / (tp_m + fn_m)
f1_manual = 2 * (precision_manual * recall_manual) / (precision_manual + recall_manual)

print("=== PRECISION & RECALL (Manual) ===\n")
print(f"TP={tp_m}, FP={fp_m}, FN={fn_m}\n")
print(f"Precision = TP/(TP+FP) = {tp_m}/({tp_m}+{fp_m}) = {precision_manual:.3f}")
print(f"Recall    = TP/(TP+FN) = {tp_m}/({tp_m}+{fn_m}) = {recall_manual:.3f}")
print(f"F1-Score  = 2*P*R/(P+R) = {f1_manual:.3f}")

=== PRECISION & RECALL (Manual) ===

TP=17, FP=5, FN=0

Precision = TP/(TP+FP) = 17/(17+5) = 0.773
Recall    = TP/(TP+FN) = 17/(17+0) = 1.000
F1-Score  = 2*P*R/(P+R) = 0.872


In [83]:
# sklearn verification
print("\nsklearn verification:")
print(f"  precision_score: {precision_score(y_te, y_pred):.3f}")
print(f"  recall_score:    {recall_score(y_te, y_pred):.3f}")
print(f"  f1_score:        {f1_score(y_te, y_pred):.3f}")


sklearn verification:
  precision_score: 0.773
  recall_score:    1.000
  f1_score:        0.872


### Business Tradeoff: When to Prioritize Precision vs Recall

| Scenario | Prioritize | Why | Cost of FN | Cost of FP |
|----------|-----------|-----|------------|------------|
| Cancer screening | **RECALL** | Missing cancer = death | Person dies | Extra tests (scary but safe) |
| Spam filter | **PRECISION** | Blocking real email = lost business | See some spam | Miss important email |
| Fraud detection | **RECALL** | Missing fraud = financial loss | Fraudster escapes | Investigate innocent person |
| Hiring (resume filter) | **RECALL** | Missing talent = opportunity cost | Lose great candidate | Interview extra people |

In [84]:
box("danger", "Accuracy is Misleading on Imbalanced Data!",
    "If 99% of emails are NOT spam, predicting 'not spam' for EVERY email gives "
    "99% accuracy — but catches ZERO spam (recall = 0%)!<br><br>"
    "Accuracy hides the problem. The confusion matrix, precision, and recall reveal it.<br>"
    "<b>Always check precision and recall for classification, not just accuracy.</b>")

In [85]:
box("definition", "F1-Score",
    "F1 = 2 * (Precision * Recall) / (Precision + Recall)<br><br>"
    "The <b>harmonic mean</b> of precision and recall — balances both into one number.<br>"
    "F1 is high ONLY when BOTH precision and recall are high.<br>"
    "Use when you need a single metric that respects the tradeoff.")

---

<div style="background-color:#ED1C24; color:white; padding:15px 25px; border-radius:8px; margin:20px 0;">
<h2 style="color:white; margin:0;">Part 6: Threshold Tuning & Decision Boundary</h2>
</div>

### The 0.5 Threshold is a Default, NOT a Law

Logistic regression outputs **probabilities**. To get a label, we apply a threshold:
- Default: if P(y=1) >= 0.5 → predict 1, else predict 0
- But **you can choose any threshold** based on business needs:
  - Lower threshold (0.3) → predict more 1's → **higher recall, lower precision**
  - Higher threshold (0.7) → predict fewer 1's → **lower recall, higher precision**

Let's see how changing the threshold affects precision, recall, and F1:

In [86]:
# Threshold effect on metrics
y_proba_1 = model.predict_proba(X_te)[:, 1]
thresholds = [0.3, 0.5, 0.7]

print("=== THRESHOLD TUNING ===\n")
print(f"{'Threshold':>10} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Pred 1s':>10}")
print("-" * 52)
for t in thresholds:
    yp = (y_proba_1 >= t).astype(int)
    p = precision_score(y_te, yp, zero_division=0)
    r = recall_score(y_te, yp, zero_division=0)
    f = f1_score(y_te, yp, zero_division=0)
    print(f"{t:>10.1f} {p:>10.3f} {r:>10.3f} {f:>10.3f} {yp.sum():>10d}")

=== THRESHOLD TUNING ===

 Threshold  Precision     Recall         F1    Pred 1s
----------------------------------------------------
       0.3      0.630      1.000      0.773         27
       0.5      0.773      1.000      0.872         22
       0.7      0.812      0.765      0.788         16


The interactive chart below shows the **Precision-Recall-F1 tradeoff** as a function of threshold. This is a **signature visualization** — it shows the central tension of classification.

In [87]:
# Helper: compute metrics across thresholds
def compute_pr_curve(y_true, y_proba):
    ts = np.linspace(0.05, 0.95, 50)
    ps, rs, fs = [], [], []
    for t in ts:
        yp = (y_proba >= t).astype(int)
        ps.append(precision_score(y_true, yp, zero_division=0))
        rs.append(recall_score(y_true, yp, zero_division=0))
        fs.append(f1_score(y_true, yp, zero_division=0))
    return ts, np.array(ps), np.array(rs), np.array(fs)

In [88]:
# INTERACTIVE: Precision/Recall/F1 vs Threshold (SIGNATURE)
ts, ps, rs, fs = compute_pr_curve(y_te, y_proba_1)
fig = go.Figure()
fig.add_trace(go.Scatter(x=ts, y=ps, mode="lines", name="Precision",
    line=dict(color="#3498DB", width=2), hovertemplate="Thresh=%{x:.2f}<br>Prec=%{y:.3f}<extra></extra>"))
fig.add_trace(go.Scatter(x=ts, y=rs, mode="lines", name="Recall",
    line=dict(color="#2ECC71", width=2), hovertemplate="Thresh=%{x:.2f}<br>Recall=%{y:.3f}<extra></extra>"))
fig.add_trace(go.Scatter(x=ts, y=fs, mode="lines", name="F1-Score",
    line=dict(color="#FF9100", width=2, dash="dash"), hovertemplate="Thresh=%{x:.2f}<br>F1=%{y:.3f}<extra></extra>"))
fig.add_vline(x=0.5, line_dash="dot", line_color="gray", annotation_text="Default (0.5)")
best_t = ts[np.argmax(fs)]
fig.add_vline(x=best_t, line_dash="dash", line_color="#ED1C24", annotation_text=f"Best F1 (t={best_t:.2f})")
fig.update_layout(title="Precision / Recall / F1 vs Threshold", width=900, height=480,
    xaxis_title="Decision Threshold", yaxis_title="Score", yaxis=dict(range=[0, 1.05]))
fig.show()

In [89]:
box("tip", "Threshold Tuning Rule of Thumb",
    "<b>Lower threshold (e.g., 0.3):</b> predict more positives -> higher recall, lower precision. "
    "Use for cancer screening (catch every case).<br>"
    "<b>Higher threshold (e.g., 0.7):</b> predict fewer positives -> higher precision, lower recall. "
    "Use for spam filtering (never block a real email).")

### Decision Boundary Visualization

The chart below shows a 2D scatter of two classes with the logistic regression **decision boundary** (the line where P(y=1) = 0.5). Points near the boundary are uncertain; points far away are confident.

In [90]:
# INTERACTIVE: 2D Decision Boundary
xx = np.linspace(X_cls[:,0].min()-1, X_cls[:,0].max()+1, 200)
yy = np.linspace(X_cls[:,1].min()-1, X_cls[:,1].max()+1, 200)
XX, YY = np.meshgrid(xx, yy)
Z = model.predict_proba(np.c_[XX.ravel(), YY.ravel()])[:,1].reshape(XX.shape)

fig = go.Figure()
fig.add_trace(go.Contour(x=xx, y=yy, z=Z, contours=dict(start=0.5, end=0.5, size=0.1),
    line=dict(color="#ED1C24", width=3), showscale=False, name="Boundary (P=0.5)"))


In [91]:
for cls, color, name in [(0,"#3498DB","Class 0"), (1,"#2ECC71","Class 1")]:
    mask = y_cls == cls
    probs = model.predict_proba(X_cls[mask])[:,1]
    hover = [f"Class: {cls}<br>P(1)={p:.3f}" for p in probs]
    fig.add_trace(go.Scatter(x=X_cls[mask,0], y=X_cls[mask,1], mode="markers",
        marker=dict(color=color, size=7, opacity=0.7), name=name,
        text=hover, hovertemplate="%{text}<extra></extra>"))
fig.update_layout(title="Decision Boundary: Where the Model Switches Classes",
    xaxis_title="Feature 1", yaxis_title="Feature 2", width=800, height=550)
fig.show()

---

<div style="background-color:#ED1C24; color:white; padding:15px 25px; border-radius:8px; margin:20px 0;">
<h2 style="color:white; margin:0;">Part 7: Mini-Project — Breast Cancer Classification</h2>
</div>

Build a classifier to detect breast cancer using sklearn's built-in dataset (569 samples, 30 features). The target: **malignant** (cancer) vs **benign** (no cancer).

**The critical question:** For cancer detection, which metric matters most?

In [92]:
# Load breast cancer dataset
data = load_breast_cancer()
X_bc = pd.DataFrame(data.data, columns=data.feature_names)
y_bc = data.target  # 0=malignant, 1=benign

print("=== BREAST CANCER DATASET ===")
print(f"Samples: {len(y_bc)}")
print(f"Features: {X_bc.shape[1]}")
print(f"Classes: {data.target_names}")
print(f"Malignant: {(y_bc==0).sum()}, Benign: {(y_bc==1).sum()}")

=== BREAST CANCER DATASET ===
Samples: 569
Features: 30
Classes: ['malignant' 'benign']
Malignant: 212, Benign: 357


In [93]:
# Pipeline: StandardScaler + LogisticRegression
X_bc_tr, X_bc_te, y_bc_tr, y_bc_te = train_test_split(X_bc, y_bc, test_size=0.2, random_state=42)

pipe = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=5000, random_state=42))])
pipe.fit(X_bc_tr, y_bc_tr)
y_bc_pred = pipe.predict(X_bc_te)

print("=== MINI-PROJECT RESULTS ===\n")
print(classification_report(y_bc_te, y_bc_pred, target_names=data.target_names))

=== MINI-PROJECT RESULTS ===

              precision    recall  f1-score   support

   malignant       0.98      0.95      0.96        43
      benign       0.97      0.99      0.98        71

    accuracy                           0.97       114
   macro avg       0.97      0.97      0.97       114
weighted avg       0.97      0.97      0.97       114



In [94]:
# Confusion matrix for breast cancer
cm_bc = confusion_matrix(y_bc_te, y_bc_pred)
tn_bc, fp_bc, fn_bc, tp_bc = cm_bc.ravel()

fig = go.Figure(go.Heatmap(z=cm_bc, x=["Pred Malignant","Pred Benign"],
    y=["True Malignant","True Benign"], colorscale="Blues", showscale=False,
    text=[[f"{tn_bc}\n(TN)", f"{fp_bc}\n(FP)"], [f"{fn_bc}\n(FN)", f"{tp_bc}\n(TP)"]],
    texttemplate="%{text}", textfont=dict(size=16)))
fig.update_layout(title="Breast Cancer Classification — Confusion Matrix",
    width=500, height=450)
fig.show()

In [95]:
# Key question: recall for malignant class
recall_malignant = tn_bc / (tn_bc + fp_bc)  # malignant is class 0
print(f"Recall for malignant (cancer): {recall_score(y_bc_te, y_bc_pred, pos_label=0):.3f}")
print(f"Precision for malignant:       {precision_score(y_bc_te, y_bc_pred, pos_label=0):.3f}")
print(f"\nMissed cancer cases (FN for malignant = FP here): {fp_bc}")
print(f"False alarms (FP for malignant = FN here): {fn_bc}")

Recall for malignant (cancer): 0.953
Precision for malignant:       0.976

Missed cancer cases (FN for malignant = FP here): 2
False alarms (FP for malignant = FN here): 1


In [96]:
box("tip", "For Cancer Detection: Maximize Recall",
    "Missing a cancer case (False Negative) is FAR worse than a false alarm (False Positive).<br>"
    "A false positive leads to more tests — scary but the patient is fine.<br>"
    "A false negative means no treatment — the cancer grows.<br><br>"
    "<b>For medical applications, recall for the dangerous class is the #1 priority.</b>")

---

## Exercise: Self-Check Questions

1. Why does LinearRegression fail for binary classification? What specific problem occurs?
2. What is sigma(0), and what does it mean for prediction at the default threshold?
3. If a model has precision=0.95 and recall=0.40, what is happening in plain English?
4. For fraud detection, should you prioritize precision or recall? Why?
5. If you lower the decision threshold from 0.5 to 0.3, what happens to precision and recall?

<details>
<summary><b>Click for answers</b></summary>

1. LinearRegression outputs values outside [0,1] (e.g., -0.2 or 1.3), which are meaningless as probabilities.
2. sigma(0) = 0.5 — the model is maximally uncertain. At threshold 0.5, this is the tipping point.
3. The model rarely predicts positive (high precision = when it does, it is usually right), but it MISSES most actual positives (low recall = 60% of real cases go undetected).
4. **Recall** — missing a fraud case (FN) means the fraudster escapes with stolen money. A false alarm (FP) just means investigating an innocent transaction.
5. Recall INCREASES (more positives predicted = more real positives caught), Precision DECREASES (more false positives in the mix).

</details>

---

## Summary & Key Takeaways

<div style="background: #EAFAF1; border-left: 5px solid #2ECC71; padding: 15px; margin: 15px 0;">
<strong>Key Takeaways:</strong>
<ol>
<li><strong>LinearRegression fails</strong> for classification — outputs outside [0,1]</li>
<li><strong>Sigmoid</strong> sigma(z) = 1/(1+e^-z) squashes any input to [0,1]</li>
<li>Logistic regression outputs <strong>probabilities</strong>, then applies threshold for labels</li>
<li><strong>predict()</strong> returns labels; <strong>predict_proba()</strong> returns probabilities</li>
<li><strong>Log loss</strong> penalizes confident wrong predictions catastrophically</li>
<li><strong>Confusion matrix:</strong> TP, FP (Type I error), FN (Type II error), TN</li>
<li><strong>Precision</strong> = "how pure are my positive predictions?" = TP/(TP+FP)</li>
<li><strong>Recall</strong> = "how many actual positives did I find?" = TP/(TP+FN)</li>
<li><strong>Threshold 0.5</strong> is default, NOT fixed — tune based on business needs</li>
<li>Cancer screening -> maximize <strong>recall</strong>. Spam filter -> maximize <strong>precision</strong></li>
</ol>
</div>

**Next Session:** Session 9.3 — Data Collection. Now you can build models — but where does the DATA come from? You will learn to collect data from APIs and web scraping.

---
<div style="text-align:center; color:#666; padding:20px;"><em>Vishlesan i-Hub IIT Patna x Masai School</em></div>

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

In [107]:
# Point A -> 7
sigmoid(7)

np.float64(0.9990889488055994)

In [108]:
# Point B -> 2
sigmoid(2)

np.float64(0.8807970779778823)

In [109]:
# Point C -> -4
sigmoid(-4)

np.float64(0.01798620996209156)

In [110]:
# Point D -> -9
sigmoid(-9)

np.float64(0.00012339457598623172)

In [111]:
sigmoid(0)

np.float64(0.5)

In [122]:
def log_loss(y,p):
  return -((y*np.log2(p))+((1-y)*np.log2(1-p)))

In [131]:
log_loss(1,0.99)

np.float64(0.014499569695115089)

In [132]:
log_loss(0,0.001)

np.float64(0.0014434168696687186)

In [133]:
log_loss(0,0.99)

np.float64(6.6438561897747235)

In [134]:
log_loss(1,0.001)

np.float64(9.965784284662087)